# 🧠 Deep Learning with Fashion-MNIST

> **Master neural networks and CNNs with practical image classification**

This notebook provides hands-on experience with deep learning using TensorFlow/Keras on the Fashion-MNIST dataset.

## 🎯 Learning Objectives

By the end of this notebook, you will:
- **Build** neural networks from scratch with TensorFlow/Keras
- **Implement** Convolutional Neural Networks (CNNs)
- **Master** data augmentation and regularization
- **Optimize** deep learning models
- **Deploy** trained models for inference

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print("✅ All imports successful!")

## 📊 Data Loading and Exploration

In [ ]:
# Load Fashion-MNIST dataset
(x_train, y_train), (x_test, y_test) = keras.datasets.fashion_mnist.load_data()

# Class names
class_names = ['T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat',
               'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot']

print(f"Training data shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Test labels shape: {y_test.shape}")
print(f"Number of classes: {len(class_names)}")

# Visualize sample images
plt.figure(figsize=(15, 6))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i], cmap='gray')
    plt.title(f'{class_names[y_train[i]]}')
    plt.axis('off')
plt.suptitle('Fashion-MNIST Sample Images')
plt.tight_layout()
plt.show()

# Data preprocessing
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print("✅ Data loaded and preprocessed!")

## 🧠 Neural Network Implementation

In [ ]:
def create_dense_model(input_shape, num_classes, hidden_layers=[128, 64]):
    """
    Create a dense neural network model
    """
    model = keras.Sequential([
        layers.Flatten(input_shape=input_shape),
        layers.Dense(hidden_layers[0], activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(hidden_layers[1], activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def create_cnn_model(input_shape, num_classes):
    """
    Create a Convolutional Neural Network model
    """
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation='relu'),
        
        layers.Flatten(),
        layers.Dense(64, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

def create_advanced_cnn_model(input_shape, num_classes):
    """
    Create an advanced CNN with batch normalization and more layers
    """
    model = keras.Sequential([
        layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.Conv2D(32, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),
        
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation='softmax')
    ])
    
    return model

print("✅ Model creation functions defined!")

In [ ]:
# Create and compile models
input_shape = (28, 28)
cnn_input_shape = (28, 28, 1)
num_classes = 10

# Reshape data for CNN
x_train_cnn = x_train.reshape(-1, 28, 28, 1)
x_test_cnn = x_test.reshape(-1, 28, 28, 1)

# Create models
dense_model = create_dense_model(input_shape, num_classes)
cnn_model = create_cnn_model(cnn_input_shape, num_classes)
advanced_cnn_model = create_advanced_cnn_model(cnn_input_shape, num_classes)

# Compile models
for model in [dense_model, cnn_model, advanced_cnn_model]:
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

# Display model architectures
print("=== DENSE MODEL ARCHITECTURE ===")
dense_model.summary()

print("\n=== CNN MODEL ARCHITECTURE ===")
cnn_model.summary()

## 🚀 Model Training and Comparison

In [ ]:
class ModelTrainer:
    def __init__(self):
        self.histories = {}
        self.models = {}
    
    def train_model(self, name, model, x_train, y_train, x_val, y_val, epochs=10):
        """Train a model and store history"""
        print(f"🚀 Training {name}...")
        
        # Define callbacks
        callbacks = [
            keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=2)
        ]
        
        # Train model
        history = model.fit(
            x_train, y_train,
            batch_size=128,
            epochs=epochs,
            validation_data=(x_val, y_val),
            callbacks=callbacks,
            verbose=1
        )
        
        self.histories[name] = history
        self.models[name] = model
        
        return history
    
    def plot_training_history(self):
        """Plot training history for all models"""
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        
        # Plot accuracy
        for name, history in self.histories.items():
            axes[0, 0].plot(history.history['accuracy'], label=f'{name} - Train')
            axes[0, 0].plot(history.history['val_accuracy'], label=f'{name} - Val', linestyle='--')
        
        axes[0, 0].set_title('Model Accuracy')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].legend()
        axes[0, 0].grid(True)
        
        # Plot loss
        for name, history in self.histories.items():
            axes[0, 1].plot(history.history['loss'], label=f'{name} - Train')
            axes[0, 1].plot(history.history['val_loss'], label=f'{name} - Val', linestyle='--')
        
        axes[0, 1].set_title('Model Loss')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Loss')
        axes[0, 1].legend()
        axes[0, 1].grid(True)
        
        plt.tight_layout()
        plt.show()
    
    def evaluate_models(self, x_test, y_test):
        """Evaluate all trained models"""
        results = {}
        
        for name, model in self.models.items():
            test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
            results[name] = {'loss': test_loss, 'accuracy': test_acc}
            print(f"{name}: Test Accuracy = {test_acc:.4f}, Test Loss = {test_loss:.4f}")
        
        return results

# Initialize trainer
trainer = ModelTrainer()

# Split training data for validation
val_size = 10000
x_val = x_train[-val_size:]
y_val = y_train[-val_size:]
x_train_split = x_train[:-val_size]
y_train_split = y_train[:-val_size]

x_val_cnn = x_train_cnn[-val_size:]
x_train_cnn_split = x_train_cnn[:-val_size]

# Train models
trainer.train_model('Dense NN', dense_model, x_train_split, y_train_split, x_val, y_val, epochs=15)
trainer.train_model('CNN', cnn_model, x_train_cnn_split, y_train_split, x_val_cnn, y_val, epochs=15)

# Plot training history
trainer.plot_training_history()

# Evaluate models
print("\n=== MODEL EVALUATION ===")
dense_results = trainer.evaluate_models(x_test, y_test)
cnn_results = trainer.models['CNN'].evaluate(x_test_cnn, y_test, verbose=0)
print(f"CNN: Test Accuracy = {cnn_results[1]:.4f}, Test Loss = {cnn_results[0]:.4f}")

## 🎯 Practice Problems

### **Problem 1: Data Augmentation**
Implement data augmentation to improve model performance.

In [ ]:
def create_data_augmentation():
    """
    Create data augmentation pipeline
    
    Include:
    - Random rotation
    - Random zoom
    - Random width/height shift
    - Random horizontal flip
    
    Returns:
    tf.keras.preprocessing.image.ImageDataGenerator
    """
    # Your code here
    pass

def train_with_augmentation(model, x_train, y_train, x_val, y_val):
    """
    Train model with data augmentation
    
    Returns:
    Training history
    """
    # Your code here
    pass

# Test your implementation
# augmented_model = create_cnn_model(cnn_input_shape, num_classes)
# augmented_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
# history = train_with_augmentation(augmented_model, x_train_cnn_split, y_train_split, x_val_cnn, y_val)

### **Problem 2: Transfer Learning**
Implement transfer learning using a pre-trained model.

In [ ]:
def create_transfer_learning_model(input_shape, num_classes, base_model_name='VGG16'):
    """
    Create transfer learning model
    
    Steps:
    1. Load pre-trained base model
    2. Freeze base model layers
    3. Add custom classification head
    4. Compile model
    
    Returns:
    tf.keras.Model
    """
    # Your code here
    pass

# Test your implementation
# Note: You'll need to resize images to 32x32x3 for VGG16
# transfer_model = create_transfer_learning_model((32, 32, 3), num_classes)

## 🎯 Key Takeaways

1. **CNNs** significantly outperform dense networks for image data
2. **Batch normalization** and **dropout** help prevent overfitting
3. **Data augmentation** improves generalization
4. **Transfer learning** can accelerate training and improve performance
5. **Callbacks** enable automatic training optimization

## 🔗 Next Steps

1. **Complete the practice problems** above
2. **Experiment with different architectures**
3. **Move to the next notebook**: Model Deployment

---

**Excellent deep learning skills!** 🎉 You can now build and train neural networks for image classification.